## SAX, Hierarchical clustering

Main pipeline. See README for details.

## 0. Configuration

Edit paths and headline modeling choices here. The rest of the notebook runs top-to-bottom.

In [1]:
from __future__ import annotations

import json
import os
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.signal import savgol_filter
from scipy.spatial.distance import squareform, pdist
from scipy.stats import norm
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

from tslearn.piecewise import SymbolicAggregateApproximation, PiecewiseAggregateApproximation

# -----------------------------------------------------------------------------
# PATHS -- environment-variable overridable
# -----------------------------------------------------------------------------
# relative defaults so the
# notebook runs out of the box for any collaborator who clones the repo.
# Override via a .env file or shell export, e.g.:
#   export SAX_DATA_DIR=/path/to/data_events
#   export SAX_OUTPUT_DIR=/path/to/output/run_01
# DATA_DIR = Path(os.environ.get("SAX_DATA_DIR", "./data/data_events"))
# OUTPUT_DIR = Path(os.environ.get("SAX_OUTPUT_DIR", "./output/sax_run"))

# SP500_PATH = Path(os.environ.get("SAX_SP500_PATH", "./data/shock_detection/SP500_data.csv"))
# DJ_PATH = Path(os.environ.get("SAX_DJ_PATH", "./data/shock_detection/DOWJONES_data.csv"))
# NASDAQ_PATH = Path(os.environ.get("SAX_NASDAQ_PATH", "./data/shock_detection/NASDAQ100_data.csv"))
# RUSSELL_PATH = Path(os.environ.get("SAX_RUSSELL_PATH", "./data/shock_detection/RUSSELL2000_data.csv"))

DATA_DIR = Path(r"C:\Python\CSUREMM\data\processed_gt")
OUTPUT_DIR = Path(r"C:\Python\CSUREMM\output")

STITCHED_SUBDIR = "stitched"
STITCHED_GLOB = "gt_fixed02*.csv"

START_DATE = "2022-01-01"
END_DATE = "2026-05-31"

# -----------------------------------------------------------------------------
# Data-specific filtering and preprocessing
# -----------------------------------------------------------------------------
MAX_ZERO_SHARE = 0.50
MAX_MISSING_SHARE = 0.02
MAX_INTERPOLATE_GAP = 7

DENOISE_WINDOW = 9
DENOISE_POLYORDER = 2
DETREND_WINDOW = 91

# -----------------------------------------------------------------------------
# SAX and hierarchical clustering
# -----------------------------------------------------------------------------
MINDIST_CHUNK_SIZE = 256
K_RANGE = range(2, 13)
RANDOM_STATE = 42
N_SEGMENTS = 96                   
ALPHABET_SIZE = 9                  # same rationale as N_SEGMENTS above
FINAL_K = 10                       # primary k: see candidate_k_interpretation_summary.csv
CANDIDATE_K_VALUES = [7, 8, 9, 10, 11, 12, 13]  # k values fit and compared side-by-side in Steps 4-5

if FINAL_K not in CANDIDATE_K_VALUES:
    raise ValueError(
        f"FINAL_K={FINAL_K} must be one of CANDIDATE_K_VALUES={CANDIDATE_K_VALUES}. "
        "Add it to the list, or change FINAL_K to a value already being compared."
    )

# PAA BREAKPOINT_MODE = "empirical" derives breakpoints from the observed distribution
# instead of theoretical N(0,1) quantiles. Set back to "gaussian" to reproduce
# the original SAX behavior.
BREAKPOINT_MODE = "empirical"       # {"gaussian", "empirical"}

# This grid is for a diagnostic that reports how tight MINDIST is against true Euclidean distance
# at each (w, a) combination to justify the parameter selection
SEGMENT_GRID = [48, 64, 96, 128]
ALPHABET_GRID = [5, 7, 9, 11]
ABLATION_SAMPLE_SIZE = 150          # terms subsampled for the (w, a) tightness check

# Default to ward linkage, but empirically compare all four candidates
LINKAGE_METHOD = "ward"           # {"single", "average", "complete", "ward"}
LINKAGE_CANDIDATES = ["single", "average", "complete", "ward"]

# MINDIST produces exact/near-exact ties for genuinely distinct series
# (a direct consequence of quantization). Within TIE_EPS of each other, merge
# order is broken using the true Euclidean distance on the underlying series
# instead of being left to array/index order inside scipy's linkage.
TIE_EPS = 1e-6

# Step 5 also logs the stability/silhouette-recommended k (subject to this
# floor) alongside FINAL_K for comparison; it never overrides FINAL_K.
MIN_ACCEPTABLE_CLUSTER_SIZE = 15

# Stability: each iteration draws two overlapping subsamples and compares labels
STABILITY_SUBSAMPLES = 1000
STABILITY_FRACTION = 0.80

# Cluster-index construction (used in Section 5)
CORE_STABILITY_QUANTILE = 0.75
MIN_CORE_TERMS = 10
TOP_N_CORE_TERMS = 10

SUBDIRS = [
    "00_provenance",
    "01_preprocessing",
    "02_sax",
    "03_distance",
    "04_clustering",
    "05_validation",
]

for sub in SUBDIRS:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())

Output directory: C:\Python\CSUREMM\output


## 1. Load, filter, denoise, detrend, and normalize

In [2]:
def clean_term_from_folder(folder: Path) -> str:
    return folder.name.replace("_", " ").strip()


def find_value_column(df: pd.DataFrame) -> str:
    date_like = {"time", "date", "week"}
    candidates = [c for c in df.columns if c.lower() not in date_like]

    if not candidates:
        raise ValueError("No value column found.")

    numeric = [
        c for c in candidates
        if pd.api.types.is_numeric_dtype(pd.to_numeric(df[c], errors="coerce"))
    ]

    return numeric[0] if numeric else candidates[0]


def max_consecutive_missing(s: pd.Series) -> int:
    missing = s.isna()

    if not missing.any():
        return 0

    groups = (missing != missing.shift()).cumsum()

    return int(
        missing
        .groupby(groups)
        .sum()
        .max()
    )


def load_one_series(folder: Path) -> pd.Series:
    stitched_dir = folder / STITCHED_SUBDIR
    files = sorted(stitched_dir.glob(STITCHED_GLOB)) if stitched_dir.exists() else []

    if not files:
        raise FileNotFoundError("missing stitched file")

    path = files[-1]
    df = pd.read_csv(path)

    date_col = next(
        (c for c in df.columns if c.lower() in {"time", "date", "week"}),
        df.columns[0],
    )

    value_col = find_value_column(df)

    dates = pd.to_datetime(df[date_col], errors="coerce")
    values = pd.to_numeric(df[value_col], errors="coerce")

    s = pd.Series(
        values.values,
        index=dates,
        name=clean_term_from_folder(folder),
    )

    s = s[~s.index.isna()]
    s = s[~s.index.duplicated(keep="last")]
    s = s.sort_index()

    s = s.loc[START_DATE:END_DATE]

    full_index = pd.date_range(
        START_DATE,
        END_DATE,
        freq="D",
    )

    return s.reindex(full_index)


def build_panel(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    series = {}

    folders = sorted([p for p in data_dir.iterdir() if p.is_dir()])

    expected_days = len(
        pd.date_range(
            START_DATE,
            END_DATE,
            freq="D",
        )
    )

    for folder in folders:
        term = clean_term_from_folder(folder)

        try:
            s = load_one_series(folder)

            observed_share = float(s.notna().mean())
            missing_share = float(s.isna().mean())
            zero_share = float((s.dropna() == 0).mean())
            longest_missing_gap = max_consecutive_missing(s)
            n_unique = int(s.nunique(dropna=True))

            reason = "kept"

            if zero_share > MAX_ZERO_SHARE:
                reason = "high_zero_share"
            elif missing_share > MAX_MISSING_SHARE:
                reason = "high_missing_share"
            elif longest_missing_gap > MAX_INTERPOLATE_GAP:
                reason = "long_missing_gap"
            elif n_unique <= 2:
                reason = "near_constant"

            rows.append({
                "term": term,
                "n_days": expected_days,
                "n_observed": int(s.notna().sum()),
                "observed_share": observed_share,
                "missing_share": missing_share,
                "zero_share": zero_share,
                "longest_missing_gap": longest_missing_gap,
                "n_unique": n_unique,
                "drop_reason": reason,
            })

            if reason == "kept":
                series[term] = s

        except Exception as e:
            rows.append({
                "term": term,
                "n_days": expected_days,
                "n_observed": 0,
                "observed_share": 0.0,
                "missing_share": 1.0,
                "zero_share": np.nan,
                "longest_missing_gap": np.nan,
                "n_unique": 0,
                "drop_reason": str(e),
            })

    panel = pd.DataFrame(series).sort_index()
    funnel = pd.DataFrame(rows).sort_values(["drop_reason", "term"])

    return panel, funnel


def interpolate_small_gaps(s: pd.Series) -> pd.Series:
    return (
        s.astype(float)
        .interpolate(
            method="time",
            limit=MAX_INTERPOLATE_GAP,
            limit_direction="both",
        )
    )


def denoise_series(s: pd.Series) -> pd.Series:
    x = s.astype(float)
    valid = x.dropna()

    if len(valid) < DENOISE_WINDOW or DENOISE_WINDOW % 2 == 0:
        return x

    filled = x.interpolate(
        method="time",
        limit_direction="both",
    )

    return pd.Series(
        savgol_filter(
            filled,
            DENOISE_WINDOW,
            DENOISE_POLYORDER,
        ),
        index=x.index,
        name=x.name,
    )


def detrend_series(s: pd.Series) -> pd.Series:
    trend = s.rolling(
        DETREND_WINDOW,
        center=True,
        min_periods=max(14, DETREND_WINDOW // 4),
    ).median()

    return s - trend


def robust_zscore(
    s: pd.Series,
    mad_floor_frac: float = 0.05,
    mad_ratio_threshold: float = 0.02,
    clip_mad_multiple: float = 4.0,
) -> pd.Series:
    x = s.astype(float)

    med = x.median(skipna=True)
    mad = (x - med).abs().median(skipna=True)

    if pd.isna(mad) or mad == 0:
        std = x.std(skipna=True)

        if pd.isna(std) or std == 0:
            return pd.Series(
                np.zeros(len(x)),
                index=x.index,
                name=x.name,
            )

        return (
            (x - x.mean(skipna=True)) / std
        ).rename(x.name)

    p05, p95 = x.quantile(0.05), x.quantile(0.95)
    wide_spread = (p95 - p05) / 3.29

    mad_ratio = (
        mad / wide_spread
        if wide_spread > 0
        else 1.0
    )

    is_degenerate = mad_ratio < mad_ratio_threshold

    if not is_degenerate:
        return (
            0.6745 * (x - med) / mad
        ).rename(x.name)

    mad_floored = (
        max(mad, mad_floor_frac * wide_spread)
        if wide_spread > 0
        else mad
    )

    z = (
        0.6745 * (x - med) / mad_floored
    ).rename(x.name)

    return z.clip(
        lower=-clip_mad_multiple,
        upper=clip_mad_multiple,
    )


def preprocess_panel(
    panel: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:

    stages = {}

    stages["filled"] = panel.apply(
        interpolate_small_gaps,
        axis=0,
    )

    stages["denoised"] = stages["filled"].apply(
        denoise_series,
        axis=0,
    )

    stages["detrended"] = stages["denoised"].apply(
        detrend_series,
        axis=0,
    )

    stages["normalized"] = stages["detrended"].apply(
        robust_zscore,
        axis=0,
    )

    return stages["normalized"], stages


def save_panel_stages(panel_raw: pd.DataFrame, stages: dict[str, pd.DataFrame]) -> None:
    panel_raw.to_csv(OUTPUT_DIR / "01_preprocessing" / "panel_raw_retained.csv")
    for name, df in stages.items():
        df.to_csv(OUTPUT_DIR / "01_preprocessing" / f"panel_{name}.csv")

In [3]:
panel_raw, filtering_funnel = build_panel(DATA_DIR)
filtering_funnel.to_csv(OUTPUT_DIR / "00_provenance" / "filtering_funnel.csv", index=False)

panel_norm, stages = preprocess_panel(panel_raw)
panel_norm = panel_norm.dropna(axis=1, how="all")
panel_norm = panel_norm.loc[:, panel_norm.notna().mean() > 0.95]
stages["normalized"] = panel_norm

save_panel_stages(panel_raw, stages)

summary = pd.DataFrame({
    "n_days": [panel_norm.shape[0]],
    "n_terms_retained": [panel_norm.shape[1]],
    "start_date": [panel_norm.index.min()],
    "end_date": [panel_norm.index.max()],
})
summary.to_csv(OUTPUT_DIR / "00_provenance" / "analysis_sample_summary.csv", index=False)

print(summary.to_string(index=False))
print("\nFiltering funnel:")
print(filtering_funnel["drop_reason"].value_counts(dropna=False).to_string())

 n_days  n_terms_retained start_date   end_date
   1612               847 2022-01-01 2026-05-31

Filtering funnel:
drop_reason
kept                  847
high_zero_share       352
high_missing_share      4


In [4]:
panel_norm.isna().sum().sum() == 0

np.True_

## 2. `tslearn` SAX representation

Each retained term is represented as one SAX string. The notebook uses `tslearn.piecewise.SymbolicAggregateApproximation`. Linearly fills missing values, only for the representation step, though previous steps should have already produced a clean panel without NAs.

In [5]:
def panel_to_tslearn_array(panel: pd.DataFrame) -> np.ndarray:
    filled = panel.astype(float).interpolate("time").ffill().bfill()
    X = filled.T.to_numpy(dtype=float)
    return X[:, :, None]


def sax_to_strings(codes_2d: np.ndarray, alphabet_size: int, index) -> pd.Series:
    alphabet = np.array(list("abcdefghijklmnopqrstuvwxyz"[:alphabet_size]))
    strings = ["".join(alphabet[row]) for row in codes_2d]
    return pd.Series(strings, index=index, name="sax_string")


def sax_breakpoints(alphabet_size: int) -> np.ndarray:
    """Gaussian breakpoints used by standard SAX (theoretical N(0,1) assumption)."""
    return norm.ppf(np.arange(1, alphabet_size) / alphabet_size)


def empirical_breakpoints(paa_values: np.ndarray, alphabet_size: int) -> np.ndarray:
    """
    Data-driven analogue of sax_breakpoints(): equiprobable quantiles of the
    pooled, observed PAA-coefficient distribution rather than theoretical
    N(0,1) quantiles. Use when the Gaussian assumption doesn't fit the data
    (see the fit diagnostic immediately below).
    """
    qs = np.arange(1, alphabet_size) / alphabet_size
    return np.quantile(paa_values, qs)


def discretize_paa(paa_values_2d: np.ndarray, breakpoints: np.ndarray) -> np.ndarray:
    """Assign each PAA coefficient to a symbol index given a set of breakpoints."""
    return np.digitize(paa_values_2d, breakpoints)


# throwing away extreme values to avoid distortion in SAX representation
panel_sax = panel_norm.clip(-5, 5)
X_ts = panel_to_tslearn_array(panel_sax)

paa = PiecewiseAggregateApproximation(n_segments=N_SEGMENTS)
paa_coefs = paa.fit_transform(X_ts)[:, :, 0]          # (n_terms, n_segments)
paa_flat = paa_coefs.ravel()

# BREAKPOINT_MODE (config, Section 0) selects Gaussian vs. empirical breakpoints.
if BREAKPOINT_MODE == "empirical":
    active_breakpoints = empirical_breakpoints(paa_flat, ALPHABET_SIZE)
else:
    active_breakpoints = sax_breakpoints(ALPHABET_SIZE)

sax_code_array = discretize_paa(paa_coefs, active_breakpoints)   # (n_terms, n_segments)
sax_codes = sax_code_array[:, :, None]                            # tslearn-style shape

sax_strings = sax_to_strings(sax_code_array, ALPHABET_SIZE, panel_norm.columns)

sax_df = pd.DataFrame({
    "term": sax_strings.index,
    "sax_string": sax_strings.values,
    "n_segments": N_SEGMENTS,
    "alphabet_size": ALPHABET_SIZE,
    "breakpoint_mode": BREAKPOINT_MODE,
})
sax_df.to_csv(OUTPUT_DIR / "02_sax" / "sax_strings.csv", index=False)

symbol_counts = (
    sax_df.assign(symbol=sax_df["sax_string"].map(list))
    .explode("symbol")
    .groupby(["term", "symbol"])
    .size()
    .unstack(fill_value=0)
)
symbol_counts.to_csv(OUTPUT_DIR / "02_sax" / "sax_symbol_counts.csv")

print("SAX strings:", sax_df.shape, " | breakpoint_mode:", BREAKPOINT_MODE)
sax_df.head()

SAX strings: (847, 5)  | breakpoint_mode: empirical


,term,sax_string,n_segments,alphabet_size,breakpoint_mode
0,2020 election map,gfadaebheagifafbebhicfbbaggbggcbfcbaiiaedbiadb...,96,9,empirical
1,2020 election results,fgbgfcchieihahhcaagihhcbeefehdedecbbiidaahiahf...,96,9,empirical
2,2022 election,bcgihaaiifbaaiiceadigeeeeeeeedgchfbdhdbcghidca...,96,9,empirical
3,2022 election results,edehgfchihgcaiieecdigeeeeeeeedfdfebfgecdfeibbb...,96,9,empirical
4,2024 election,gdcfcbeedahhhbccdbfihbabghiahgggheaagigaaiiaaa...,96,9,empirical


### Justify clipping value selection at $\pm 5 \text{ SD}$ is the right approach

In [6]:
for c in [2,3,4,5]:
    pct = np.mean(np.abs(panel_norm.to_numpy()) > c)
    print(c, pct)

2 0.18607492214530338
3 0.1243272856176082
4 0.09299351674718244
5 0.07572705886488877


In [7]:
# --- Diagnostic: does the Gaussian breakpoint assumption actually fit? ---
# Excess kurtosis > 0 or a small KS p-value indicates the pooled coefficients are more peaked / non-Gaussian
# which motivates BREAKPOINT_MODE = "empirical" above.
from scipy import stats as _stats

standardized = (paa_flat - paa_flat.mean()) / paa_flat.std()
excess_kurtosis = _stats.kurtosis(paa_flat, fisher=True)   # 0 for a true Gaussian
ks_stat, ks_p = _stats.kstest(standardized, "norm")

breakpoint_fit = pd.DataFrame({
    "metric": ["excess_kurtosis", "ks_statistic", "ks_pvalue", "n_paa_coefficients", "breakpoint_mode_used"],
    "value": [excess_kurtosis, ks_stat, ks_p, len(paa_flat), BREAKPOINT_MODE],
})
breakpoint_fit.to_csv(OUTPUT_DIR / "paa_gaussian_fit_diagnostic.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(paa_flat, bins=60, density=True, alpha=0.7, label="pooled PAA coefficients")
xs = np.linspace(paa_flat.min(), paa_flat.max(), 200)
axes[0].plot(xs, norm.pdf(xs, paa_flat.mean(), paa_flat.std()), lw=2, label="fitted Gaussian")
axes[0].set_title("Pooled PAA coefficients vs Gaussian fit")
axes[0].legend(frameon=False)
_stats.probplot(paa_flat, dist="norm", plot=axes[1])
axes[1].set_title("QQ-plot vs Gaussian")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "paa_gaussian_fit_diagnostic.png", dpi=200)
plt.close(fig)

print(f"Excess kurtosis: {excess_kurtosis:.3f}  (0 = Gaussian; > 0 = more peaked than Gaussian)")
print(f"KS test vs N(0,1): stat={ks_stat:.4f}, p={ks_p:.2e}")
print(f"-> breakpoint_mode currently in use: {BREAKPOINT_MODE}")

Excess kurtosis: 3.887  (0 = Gaussian; > 0 = more peaked than Gaussian)
KS test vs N(0,1): stat=0.1457, p=0.00e+00
-> breakpoint_mode currently in use: empirical


## 3. MINDIST

The clustering distance is SAX **MINDIST**, computed from the (Gaussian or
empirical, per `BREAKPOINT_MODE`) breakpoints from Section 2.

Before committing to `N_SEGMENTS` / `ALPHABET_SIZE`, a small ablation below
reports how tight MINDIST is against the true Euclidean distance across a grid
of `(n_segments, alphabet_size)` choices -- this is a diagnostic, not an
automatic override, so the hardcoded values above can be checked against
evidence.

Linkage defaults to **ward linkage**: it is the only merge rule for which
Kleinberg-style scale invariance + richness + consistency are jointly
satisfiable once stated over the dendrogram (Zadeh & Ben-David, 2009), which
is the property MINDIST-based clustering is implicitly leaning on. All four
standard linkage methods are compared empirically in Section 4 regardless, so
this default is a starting point, not an assumption baked in silently.

MINDIST also produces exact/near-exact distance ties for genuinely distinct
series (a direct consequence of quantization); these are broken using the true
Euclidean distance on the underlying series rather than left to array order.

In [8]:
def sax_symbol_distance_table(alphabet_size: int, breakpoints: np.ndarray | None = None) -> np.ndarray:
    """
    Pairwise SAX symbol distances used in MINDIST.

    Adjacent symbols have distance 0 because their intervals touch. Non-adjacent
    symbols are separated by the gap between the relevant breakpoints. Accepts an
    explicit breakpoints array so the table reflects whichever BREAKPOINT_MODE
    (Gaussian or empirical) was actually used to build the SAX codes.
    """
    bp = sax_breakpoints(alphabet_size) if breakpoints is None else breakpoints
    table = np.zeros((alphabet_size, alphabet_size), dtype=float)

    for i in range(alphabet_size):
        for j in range(alphabet_size):
            if abs(i - j) <= 1:
                table[i, j] = 0.0
            else:
                lo, hi = sorted((i, j))
                table[i, j] = bp[hi - 1] - bp[lo]
    return table

# NOTE: the (w, a) MINDIST-tightness ablation that used to run here has moved
# to Section 06 (Validity checks), alongside the reconstruction-elbow and
# downstream-quality checks it's meant to be read together with.

In [9]:
def sax_mindist_matrix(
    sax_codes: np.ndarray,
    terms: list[str],
    n_timestamps: int,
    n_segments: int,
    alphabet_size: int,
    symbol_dist: np.ndarray,
    chunk_size: int = MINDIST_CHUNK_SIZE,
) -> pd.DataFrame:
    """
    Compute SAX MINDIST between all term-level SAX strings.

    Parameters
    ----------
    sax_codes:
        Symbol-code array, shape (n_terms, n_segments, 1).
    terms:
        Term names in the same order as sax_codes.
    n_timestamps:
        Original equal-length series length before SAX compression.
    n_segments:
        Number of SAX/PAA segments.
    alphabet_size:
        Number of SAX symbols.
    symbol_dist:
        Precomputed symbol-to-symbol distance table (reflects the breakpoints
        actually used -- Gaussian or empirical -- rather than assuming Gaussian).
    chunk_size:
        Row chunk size to keep memory use stable.

    Returns
    -------
    pd.DataFrame
        Symmetric MINDIST matrix indexed and columned by term.
    """
    codes = np.asarray(sax_codes[:, :, 0], dtype=np.int16)
    n_terms, actual_segments = codes.shape

    if actual_segments != n_segments:
        raise ValueError(f"Expected {n_segments} SAX segments, got {actual_segments}.")
    if len(terms) != n_terms:
        raise ValueError("Number of terms does not match number of SAX code rows.")

    scale = np.sqrt(n_timestamps / n_segments)
    D = np.zeros((n_terms, n_terms), dtype=np.float32)

    for start in range(0, n_terms, chunk_size):
        stop = min(start + chunk_size, n_terms)
        block = codes[start:stop]
        per_segment = symbol_dist[block[:, None, :], codes[None, :, :]]
        D[start:stop, :] = scale * np.sqrt((per_segment ** 2).sum(axis=2))

    D = 0.5 * (D + D.T)
    np.fill_diagonal(D, 0.0)
    return pd.DataFrame(D, index=terms, columns=terms)


def true_euclidean_distance_matrix(panel: pd.DataFrame) -> pd.DataFrame:
    """True (full-resolution) Euclidean distance, used only to break MINDIST ties."""
    X = panel.T.to_numpy(dtype=float)
    D = squareform(pdist(X, metric="euclidean"))
    return pd.DataFrame(D, index=panel.columns, columns=panel.columns)


def break_mindist_ties(mindist_df: pd.DataFrame, euclid_df: pd.DataFrame, eps: float = TIE_EPS) -> pd.DataFrame:
    """
    MINDIST is a many-to-one lower bound and produces exact/near-exact ties for
    genuinely distinct series (see the duplicate-value counts in the SAX
    characterization diagnostics). Within `eps` those ties are effectively
    arbitrary and would otherwise be resolved by array/index order inside
    scipy's linkage. This adds a rank-preserving nudge from the true Euclidean
    distance, strictly smaller than `eps`, so no non-tied comparison is
    reordered but tied comparisons are resolved using real signal.
    """
    M = mindist_df.to_numpy(dtype=float)
    E = euclid_df.reindex(index=mindist_df.index, columns=mindist_df.columns).to_numpy(dtype=float)

    e_rank = E / (E.max() + 1e-9)          # in [0, 1), preserves Euclidean order
    nudge = eps * e_rank                    # strictly smaller than eps
    M_broken = M + nudge
    np.fill_diagonal(M_broken, 0.0)
    M_broken = 0.5 * (M_broken + M_broken.T)
    return pd.DataFrame(M_broken, index=mindist_df.index, columns=mindist_df.columns)


def hierarchical_labels(distance_df: pd.DataFrame, k: int, method: str | None = None) -> pd.Series:
    method = method or LINKAGE_METHOD
    condensed = squareform(distance_df.to_numpy(dtype=float), checks=False)
    Z = linkage(condensed, method=method)
    labels = fcluster(Z, t=k, criterion="maxclust")
    return pd.Series(labels, index=distance_df.index, name="cluster")


symbol_dist_table = sax_symbol_distance_table(ALPHABET_SIZE, active_breakpoints)

mindist = sax_mindist_matrix(
    sax_codes=sax_codes,
    terms=sax_strings.index.to_list(),
    n_timestamps=panel_norm.shape[0],
    n_segments=N_SEGMENTS,
    alphabet_size=ALPHABET_SIZE,
    symbol_dist=symbol_dist_table,
)
mindist.to_csv(OUTPUT_DIR / "03_distance" / "sax_mindist_matrix.csv")

mindist_summary = pd.DataFrame({
    "metric": ["n_terms", "n_segments", "alphabet_size", "min_offdiag", "median_offdiag", "mean_offdiag", "max_offdiag"],
    "value": [
        mindist.shape[0],
        N_SEGMENTS,
        ALPHABET_SIZE,
        np.nanmin(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
        np.nanmedian(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
        np.nanmean(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
        np.nanmax(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy()),
    ],
})
mindist_summary.to_csv(OUTPUT_DIR / "03_distance" / "sax_mindist_summary.csv", index=False)

n_exact_ties = int(np.isclose(mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy(), 0.0, atol=1e-9).sum())
print(f"Exact-zero off-diagonal MINDIST pairs before tie-breaking: {n_exact_ties}")

euclid_true = true_euclidean_distance_matrix(panel_sax)
mindist_tiebroken_raw = break_mindist_ties(mindist, euclid_true, eps=TIE_EPS)
mindist_tiebroken = np.square(mindist_tiebroken_raw)  # convert to squared distances for ward linkage

mindist_tiebroken_raw.to_csv(OUTPUT_DIR / "03_distance" / "sax_mindist_matrix_tiebroken.csv")
mindist_tiebroken.to_csv(OUTPUT_DIR / "03_distance" / "sax_mindist_matrix_tiebroken_squared.csv")

mindist_summary

Exact-zero off-diagonal MINDIST pairs before tie-breaking: 16


,metric,value
0,n_terms,847.000000
1,n_segments,96.000000
2,alphabet_size,9.000000
3,min_offdiag,0.000000
4,median_offdiag,32.555050
5,mean_offdiag,31.949610
6,max_offdiag,54.985077


### Some diagnosis on SAX characterization

In [10]:
print("panel_norm shape:", panel_norm.shape)
print("N_SEGMENTS:", N_SEGMENTS)
print("ALPHABET_SIZE:", ALPHABET_SIZE)
print("FINAL_K:", FINAL_K)
print("NaNs:", panel_norm.isna().sum().sum())
print("Constant cols:", (panel_norm.std(axis=0) == 0).sum())

panel_norm shape: (1612, 847)
N_SEGMENTS: 96
ALPHABET_SIZE: 9
FINAL_K: 10
NaNs: 0
Constant cols: 0


In [11]:
codes = np.asarray(sax_codes[:, :, 0], dtype=int)

# 1. How many unique SAX words?
sax_words = ["".join(map(str, row)) for row in codes]
print("unique SAX words:", pd.Series(sax_words).nunique())
print(pd.Series(sax_words).value_counts().head(20))

# 2. Symbol usage
symbol_counts = pd.Series(codes.ravel()).value_counts().sort_index()
print(symbol_counts)

# 3. Per-term number of unique symbols
unique_symbols_per_term = pd.Series(
    [len(set(row)) for row in codes],
    index=sax_strings.index,
    name="n_unique_symbols"
)
print(unique_symbols_per_term.describe())
print(unique_symbols_per_term.value_counts().sort_index())

# 4. Distance distribution
offdiag = mindist.where(~np.eye(len(mindist), dtype=bool)).to_numpy().ravel()
offdiag = offdiag[~np.isnan(offdiag)]
print(pd.Series(offdiag).describe())
print(pd.Series(np.round(offdiag, 6)).value_counts().head(20))

unique SAX words: 843
434765278762088442386444444443535415642354811188478000377058100188803575828583466874443885008812    2
632521443077712231587101678076667400686008800080388000660588080367444444443434633752236744215743    2
007810088210088450187644445453687413724174871065786422671777170367444444444773573721133867203748    2
136206577531215576663704766516773321165766617047465167652112676457150565643575512145666562103746    2
650304174068505141782511066166215210880431803187087200430788171367444444444444444645545786106851    1
561652278487077200687721445473434211883007807577186211371688080178444444444543644881025772808880    1
126870088510088240386444444443627513731267832044376331141477011688410444424162266880147786008850    1
414621352177723140387031588066777701686008800186088101170688080367444444443434634862245743116861    1
047208788641226786770701777316884312277767706008662276552223687357150474625564422346765561105644    1
784164464444467676500707834543754434776884083218844016511047

In [12]:
for k in range(2, 15):
    labels = hierarchical_labels(mindist_tiebroken, k)

    print(
        k,
        labels.value_counts().values
    )

2 [565 282]
3 [565 209  73]
4 [408 209 157  73]
5 [343 209 157  73  65]
6 [343 209 115  73  65  42]
7 [343 135 115  74  73  65  42]
8 [343 135 115  74  73  42  37  28]
9 [343 115 109  74  73  42  37  28  26]
10 [227 116 115 109  74  73  42  37  28  26]
11 [227 116 115 109  74  49  42  37  28  26  24]
12 [227 115 109  88  74  49  42  37  28  28  26  24]
13 [227 109 102  88  74  49  42  37  28  28  26  24  13]
14 [227 109 102  74  67  49  42  37  28  28  26  24  21  13]


## 4. Hierarchical clustering

Fit one linkage tree from `mindist_tiebroken`, then cut it at every k in
`CANDIDATE_K_VALUES` -- cutting a fixed tree at different k is essentially
free, since scipy's `linkage()` (the expensive step) doesn't depend on k at
all. Each candidate's term assignments, cluster sizes, and dendrogram are
saved to their own `04_clustering/k_{k}/` folder, plus a side-by-side
`candidate_k_comparison.csv` for comparing structural stats at a glance.

`FINAL_K` (Step 0) selects which candidate becomes the notebook-wide primary
solution: `labels_final`, used by every downstream section (validation,
robustness, poster). Change `FINAL_K` and re-run to promote a different
candidate -- no need to touch `CANDIDATE_K_VALUES` or re-fit anything.

In [13]:
import shutil

CLUSTER_DIR = OUTPUT_DIR / "04_clustering"

if not isinstance(mindist_tiebroken, pd.DataFrame):
    raise TypeError("mindist_tiebroken must be a pandas DataFrame.")
if mindist_tiebroken.shape[0] != mindist_tiebroken.shape[1]:
    raise ValueError("mindist_tiebroken must be square.")
if not mindist_tiebroken.index.equals(mindist_tiebroken.columns):
    raise ValueError("Distance-matrix rows and columns must match in the same order.")

D_final = mindist_tiebroken.to_numpy(dtype=float)
if not np.all(np.isfinite(D_final)):
    raise ValueError("mindist_tiebroken contains NaN or infinite values.")
if not np.allclose(D_final, D_final.T, atol=1e-10):
    raise ValueError("mindist_tiebroken must be symmetric.")
if not np.allclose(np.diag(D_final), 0.0, atol=1e-10):
    raise ValueError("The diagonal of mindist_tiebroken must be zero.")

for k in CANDIDATE_K_VALUES:
    if not 2 <= k < len(mindist_tiebroken):
        raise ValueError(
            f"Every value in CANDIDATE_K_VALUES must be between 2 and "
            f"{len(mindist_tiebroken) - 1}; got k={k}."
        )

# One linkage tree for the whole notebook -- every candidate k below is just
# a different cut of this same tree, not a separate fit.
Z_SHARED = linkage(squareform(D_final, checks=False), method=LINKAGE_METHOD)


def fit_hierarchical_solution(k: int, save_dir: Path, show: bool = False) -> dict:
    """
    Cut the shared linkage tree at k, save its term assignments, cluster
    sizes, and dendrogram under `save_dir`, and return everything needed to
    interpret or compare this candidate later.
    """
    save_dir.mkdir(parents=True, exist_ok=True)

    labels = pd.Series(
        fcluster(Z_SHARED, t=k, criterion="maxclust"),
        index=mindist_tiebroken.index,
        name="cluster",
    ).astype(int)

    assignments = (
        labels.rename_axis("term").reset_index().sort_values(["cluster", "term"])
    )
    assignments.to_csv(save_dir / "cluster_assignments.csv", index=False)

    sizes = (
        labels.value_counts().sort_index()
        .rename_axis("cluster").reset_index(name="n_terms")
    )
    sizes["share_of_terms"] = sizes["n_terms"] / len(labels)
    sizes.to_csv(save_dir / "cluster_sizes.csv", index=False)

    color_threshold = Z_SHARED[-(k - 1), 2] if k > 1 else None
    fig, ax = plt.subplots(figsize=(14, 6))
    dendrogram(
        Z_SHARED, no_labels=True, color_threshold=color_threshold,
        above_threshold_color="gray", ax=ax,
    )
    ax.set_title(f"Hierarchical clustering dendrogram, k = {k}, linkage = {LINKAGE_METHOD}")
    ax.set_xlabel("Search terms")
    ax.set_ylabel("SAX MINDIST (tie-broken)")
    fig.tight_layout()
    fig.savefig(save_dir / f"dendrogram_k{k}.png", dpi=300, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)

    return {"k": k, "labels": labels, "sizes": sizes}


CLUSTERING_BY_K = {
    k: fit_hierarchical_solution(k, CLUSTER_DIR / f"k_{k}", show=(k == FINAL_K))
    for k in CANDIDATE_K_VALUES
}

candidate_k_comparison = pd.DataFrame([
    {
        "k": k,
        "n_terms": len(result["labels"]),
        "min_cluster_size": int(result["sizes"]["n_terms"].min()),
        "max_cluster_size": int(result["sizes"]["n_terms"].max()),
        "median_cluster_size": float(result["sizes"]["n_terms"].median()),
        "largest_cluster_share": float(result["sizes"]["share_of_terms"].max()),
        "is_final_k": k == FINAL_K,
    }
    for k, result in CLUSTERING_BY_K.items()
])
candidate_k_comparison.to_csv(CLUSTER_DIR / "candidate_k_comparison.csv", index=False)

# Promote the configured FINAL_K to the single, notebook-wide "primary"
# solution that every downstream section (Steps 5-7) reads from here on.
labels_final = CLUSTERING_BY_K[FINAL_K]["labels"]
cluster_sizes_final = CLUSTERING_BY_K[FINAL_K]["sizes"]

# Mirror the primary solution under flat, k-independent filenames too, purely
# for discoverability -- its authoritative copy still lives in k_{FINAL_K}/.
(
    labels_final.rename_axis("term").reset_index().sort_values(["cluster", "term"])
).to_csv(CLUSTER_DIR / "final_cluster_assignments.csv", index=False)
cluster_sizes_final.to_csv(CLUSTER_DIR / "cluster_sizes_final.csv", index=False)
shutil.copy2(
    CLUSTER_DIR / f"k_{FINAL_K}" / f"dendrogram_k{FINAL_K}.png",
    CLUSTER_DIR / f"dendrogram_k{FINAL_K}.png",
)

print(f"Fit {len(CANDIDATE_K_VALUES)} candidate solutions: {CANDIDATE_K_VALUES}")
print(f"Primary (FINAL_K={FINAL_K}): {len(labels_final):,} terms across {FINAL_K} clusters.\n")
print("Candidate comparison (structural stats only -- see Step 5 for silhouette/stability):")
print(candidate_k_comparison.to_string(index=False))

Fit 7 candidate solutions: [7, 8, 9, 10, 11, 12, 13]
Primary (FINAL_K=10): 847 terms across 10 clusters.

Candidate comparison (structural stats only -- see Step 5 for silhouette/stability):
 k  n_terms  min_cluster_size  max_cluster_size  median_cluster_size  largest_cluster_share  is_final_k
 7      847                42               343                 74.0               0.404959       False
 8      847                28               343                 73.5               0.404959       False
 9      847                26               343                 73.0               0.404959       False
10      847                26               227                 73.5               0.268005        True
11      847                24               227                 49.0               0.268005       False
12      847                24               227                 45.5               0.268005       False
13      847                13               227                 42.0             

## 5. Cluster validation and interpretation

Two independent checks, both spanning multiple k:

1. **Statistical validation across the full `K_RANGE` (2-12)** -- silhouette
   and subsample stability for every k in that range, regardless of which
   ones were fit as full candidates in Step 4. Flags whether the
   stability/silhouette-recommended k agrees with `FINAL_K`, without ever
   overwriting it.
2. **Interpretive comparison across `CANDIDATE_K_VALUES`** -- term stability,
   core terms, and a cluster summary (including each cluster's MINDIST
   margin) for every candidate fit in Step 4, reusing those labels rather
   than recomputing them. `FINAL_K`'s interpretation is promoted as the
   notebook-wide primary summary that Steps 6-7 read from.

In [15]:
VALIDATION_DIR = OUTPUT_DIR / "05_validation"

def labels_from_submatrix(
    full_distance: pd.DataFrame,
    terms: list[str],
    k: int,
    method: str | None = None,
) -> pd.Series:
    method = method or LINKAGE_METHOD
    sub_distance = full_distance.loc[terms, terms]
    return hierarchical_labels(sub_distance, k=k, method=method)


def stability_for_k(
    full_distance: pd.DataFrame,
    reference_labels: pd.Series,  # Pass the full-dataset baseline labels here
    k: int,
    n_subsamples: int,
    fraction: float,
    seed: int,
    method: str | None = None,
) -> pd.DataFrame:
    method = method or LINKAGE_METHOD
    rng = np.random.default_rng(seed)
    terms = np.asarray(full_distance.index)
    n_take = int(round(len(terms) * fraction))
    rows = []

    for iteration in range(n_subsamples):
        # 1. Take a single random subsample of the data
        sample_terms = rng.choice(terms, size=n_take, replace=False).tolist()
       
        # 2. Cluster ONLY this subsample
        sample_labels = labels_from_submatrix(
            full_distance, sample_terms, k, method=method
        )
       
        # 3. Align it directly with the master labels for those exact same terms
        aligned_master = reference_labels.loc[sample_terms]

        # 4. Compute ARI between the subsample slice and the ground truth master
        ari_score = adjusted_rand_score(sample_labels, aligned_master)

        rows.append({
            "k": k,
            "iteration": iteration,
            "n_common": len(sample_terms),
            "ari": ari_score,
        })

    return pd.DataFrame(rows)

def validation_for_k(
    full_distance: pd.DataFrame,
    labels: pd.Series,
    k: int,
) -> dict:
    D = full_distance.to_numpy(dtype=float)
    labs = labels.loc[full_distance.index].to_numpy()
    n_observed_clusters = len(np.unique(labs))

    silhouette = (
        silhouette_score(D, labs, metric="precomputed")
        if 1 < n_observed_clusters < len(labs)
        else np.nan
    )
    sizes = labels.value_counts()

    return {
        "k": int(k),
        "n_observed_clusters": int(n_observed_clusters),
        "silhouette_mindist": float(silhouette) if pd.notna(silhouette) else np.nan,
        "min_cluster_size": int(sizes.min()),
        "max_cluster_size": int(sizes.max()),
        "median_cluster_size": float(sizes.median()),
    }


# Candidate-k validation sweep. This evaluates FINAL_K but does not replace it.
validation_rows = []
stability_tables = []

for k in K_RANGE:
    if not 2 <= k < len(mindist_tiebroken):
        continue

    labels_k = hierarchical_labels(
        mindist_tiebroken,
        k=k,
        method=LINKAGE_METHOD,
    )
    validation_rows.append(validation_for_k(mindist_tiebroken, labels_k, k))
    stability_tables.append(
    stability_for_k(
        full_distance=mindist_tiebroken,
        reference_labels=labels_k,
        k=k,
        n_subsamples=STABILITY_SUBSAMPLES,
        fraction=STABILITY_FRACTION,
        seed=RANDOM_STATE + k,
        method=LINKAGE_METHOD,
    )
)

if not validation_rows:
    raise ValueError("K_RANGE contains no valid candidate values.")

validation_df = pd.DataFrame(validation_rows)
stability_raw = pd.concat(stability_tables, ignore_index=True)
stability_summary = (
    stability_raw
    .groupby("k", as_index=False)
    .agg(
        stability_mean=("ari", "mean"),
        stability_median=("ari", "median"),
        stability_p10=("ari", lambda x: x.quantile(0.10)),
        stability_p90=("ari", lambda x: x.quantile(0.90)),
        mean_common_terms=("n_common", "mean"),
        valid_stability_runs=("ari", "count"),
    )
)

cluster_selection = (
    validation_df
    .merge(stability_summary, on="k", how="left")
    .sort_values("k")
    .reset_index(drop=True)
)
cluster_selection["is_final_k"] = cluster_selection["k"].eq(FINAL_K)
cluster_selection["is_candidate_k"] = cluster_selection["k"].isin(CANDIDATE_K_VALUES)

cluster_selection.to_csv(
    VALIDATION_DIR / "cluster_selection_by_k.csv",
    index=False,
)
stability_raw.to_csv(
    VALIDATION_DIR / "subsample_stability_raw.csv",
    index=False,
)

print("Candidate cluster validation:")
print(
    cluster_selection
    .sort_values(
        ["stability_median", "silhouette_mindist"],
        ascending=[False, False],
        na_position="last",
    )
    .to_string(index=False)
)


def select_recommended_k(
    selection_df: pd.DataFrame,
    min_cluster_size: int,
) -> int:
    eligible = selection_df[
        selection_df["min_cluster_size"] >= min_cluster_size
    ]
    if eligible.empty:
        eligible = selection_df
    best = eligible.sort_values(
        ["stability_median", "silhouette_mindist"],
        ascending=[False, False],
        na_position="last",
    ).iloc[0]
    return int(best["k"])


RECOMMENDED_K = select_recommended_k(
    cluster_selection,
    MIN_ACCEPTABLE_CLUSTER_SIZE,
)

k_selection_audit = pd.DataFrame({
    "metric": [
        "configured_final_k",
        "recommended_k",
        "min_acceptable_cluster_size",
        "configured_k_matches_recommendation",
    ],
    "value": [
        FINAL_K,
        RECOMMENDED_K,
        MIN_ACCEPTABLE_CLUSTER_SIZE,
        FINAL_K == RECOMMENDED_K,
    ],
})

k_selection_audit.to_csv(
    VALIDATION_DIR / "k_selection_audit.csv",
    index=False,
)

print(
    f"\nConfigured FINAL_K={FINAL_K}; validation recommends "
    f"k={RECOMMENDED_K}. The configured baseline is not changed."
)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(
    cluster_selection["k"],
    cluster_selection["stability_median"],
    marker="o",
    label="Median subsample stability",
)
ax.plot(
    cluster_selection["k"],
    cluster_selection["silhouette_mindist"],
    marker="s",
    label="Silhouette score",
)
ax.axvline(
    FINAL_K,
    linestyle="--",
    linewidth=1.5,
    label=f"Final k = {FINAL_K}",
)
ax.set_title("Cluster validation across candidate values of k")
ax.set_xlabel("Number of clusters, k")
ax.set_ylabel("Validation score")
ax.set_xticks(cluster_selection["k"])
ax.legend(frameon=False)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(
    VALIDATION_DIR / "cluster_selection_scores.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


def term_stability_from_distance(
    full_distance: pd.DataFrame,
    final_labels: pd.Series,
) -> pd.DataFrame:
    rows = []
    aligned_labels = final_labels.loc[full_distance.index]

    for term in aligned_labels.index:
        cluster_id = aligned_labels.loc[term]
        same_terms = aligned_labels.index[
            (aligned_labels == cluster_id) & (aligned_labels.index != term)
        ]
        other_terms = aligned_labels.index[aligned_labels != cluster_id]

        intra_dist = (
            full_distance.loc[term, same_terms].mean()
            if len(same_terms) > 0
            else np.nan
        )
        between_dist = (
            full_distance.loc[term, other_terms].mean()
            if len(other_terms) > 0
            else np.nan
        )
        margin = (
            between_dist - intra_dist
            if pd.notna(intra_dist) and pd.notna(between_dist)
            else np.nan
        )

        rows.append({
            "term": term,
            "cluster": int(cluster_id),
            "mean_intra_mindist": intra_dist,
            "mean_between_mindist": between_dist,
            "mindist_margin": margin,
        })

    term_df = pd.DataFrame(rows)
    term_df["term_stability"] = (
        term_df
        .groupby("cluster")["mindist_margin"]
        .rank(pct=True, method="average")
    )
    return term_df.sort_values(
        ["cluster", "term_stability", "mindist_margin"],
        ascending=[True, False, False],
    ).reset_index(drop=True)


def select_core_terms(
    term_stability: pd.DataFrame,
    top_n: int = TOP_N_CORE_TERMS,
) -> pd.DataFrame:
    selected = []

    for cluster_id, group in term_stability.groupby("cluster"):
        group = group.sort_values(
            ["term_stability", "mindist_margin"],
            ascending=[False, False],
        ).copy()
        cutoff = group["term_stability"].quantile(CORE_STABILITY_QUANTILE)
        core = group[group["term_stability"] >= cutoff].copy()

        min_keep = min(MIN_CORE_TERMS, len(group))
        if len(core) < min_keep:
            core = group.head(min_keep).copy()
        if top_n is not None and top_n > 0:
            core = core.head(max(top_n, min_keep)).copy()

        core["core_rank"] = np.arange(1, len(core) + 1)
        selected.append(core)

    return pd.concat(selected, ignore_index=True)


def weighted_average(panel: pd.DataFrame, weights: pd.Series) -> pd.Series:
    columns = [column for column in weights.index if column in panel.columns]
    if not columns:
        return pd.Series(np.nan, index=panel.index, dtype=float)

    aligned_weights = weights.loc[columns].astype(float).fillna(0.0)
    if aligned_weights.sum() <= 0:
        aligned_weights = pd.Series(1.0 / len(columns), index=columns)
    else:
        aligned_weights = aligned_weights / aligned_weights.sum()

    return panel[columns].mul(aligned_weights, axis=1).sum(axis=1)


def interpret_solution(
    k: int,
    labels: pd.Series,
    save_dir: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build term-level stability, core terms, and a cluster summary for one
    already-fitted candidate k (from Step 4's CLUSTERING_BY_K), without
    recomputing its cluster assignments.
    """
    save_dir.mkdir(parents=True, exist_ok=True)

    term_stability_k = term_stability_from_distance(mindist, labels)
    core_terms_k = select_core_terms(term_stability_k, top_n=TOP_N_CORE_TERMS)

    term_stability_k.to_csv(save_dir / "term_stability.csv", index=False)
    core_terms_k.to_csv(save_dir / "core_terms.csv", index=False)

    summary_k = (
        term_stability_k
        .groupby("cluster", as_index=False)
        .agg(
            n_terms=("term", "size"),
            median_term_stability=("term_stability", "median"),
            mean_intra_mindist=("mean_intra_mindist", "mean"),
            mean_between_mindist=("mean_between_mindist", "mean"),
            mean_mindist_margin=("mindist_margin", "mean"),
        )
    )
    top_terms_k = (
        core_terms_k
        .sort_values(["cluster", "core_rank"])
        .groupby("cluster")["term"]
        .apply(lambda terms: ", ".join(terms.head(TOP_N_CORE_TERMS)))
    )
    summary_k["top_10_core_terms"] = summary_k["cluster"].map(top_terms_k)

    expected_sizes = labels.value_counts().sort_index()
    summary_k["expected_n_terms"] = summary_k["cluster"].map(expected_sizes)
    summary_k["size_matches_labels"] = (
        summary_k["n_terms"] == summary_k["expected_n_terms"]
    )
    if not summary_k["size_matches_labels"].all():
        raise RuntimeError(f"Cluster-summary sizes do not match labels for k={k}.")

    summary_k.to_csv(save_dir / "cluster_summary.csv", index=False)
    return term_stability_k, core_terms_k, summary_k


# Interpret every candidate fit in Step 4 -- reuses CLUSTERING_BY_K, so no
# cluster assignment is recomputed here, only its downstream interpretation.
INTERPRETATION_BY_K = {
    k: interpret_solution(
        k, CLUSTERING_BY_K[k]["labels"], VALIDATION_DIR / f"k_{k}"
    )
    for k in CANDIDATE_K_VALUES
}

candidate_k_interpretation = pd.DataFrame([
    {
        "k": k,
        "n_clusters": len(summary_k),
        "weakest_cluster_margin": float(summary_k["mean_mindist_margin"].min()),
        "strongest_cluster_margin": float(summary_k["mean_mindist_margin"].max()),
        "largest_cluster_n_terms": int(summary_k["n_terms"].max()),
        "is_final_k": k == FINAL_K,
    }
    for k, (_, _, summary_k) in INTERPRETATION_BY_K.items()
])
candidate_k_interpretation.to_csv(
    VALIDATION_DIR / "candidate_k_interpretation_summary.csv", index=False
)

print("Interpretive comparison across candidate k values:")
print(
    "(weakest_cluster_margin close to 0 or negative flags a cluster that is "
    "not well separated -- see cluster_summary.csv in that k's folder)\n"
)
print(candidate_k_interpretation.to_string(index=False))

# Promote FINAL_K's interpretation as the notebook-wide primary summary, and
# mirror it under flat, k-independent filenames for discoverability -- its
# authoritative copy still lives in k_{FINAL_K}/.
term_stability, core_terms, cluster_summary = INTERPRETATION_BY_K[FINAL_K]
term_stability.to_csv(VALIDATION_DIR / "term_stability.csv", index=False)
core_terms.to_csv(VALIDATION_DIR / "core_terms.csv", index=False)
cluster_summary.to_csv(VALIDATION_DIR / "cluster_summary.csv", index=False)

print(f"\nPrimary cluster interpretation (FINAL_K={FINAL_K}):")
print(cluster_summary.to_string(index=False))

Candidate cluster validation:
 k  n_observed_clusters  silhouette_mindist  min_cluster_size  max_cluster_size  median_cluster_size  stability_mean  stability_median  stability_p10  stability_p90  mean_common_terms  valid_stability_runs  is_final_k  is_candidate_k
12                   12            0.113176                24               227                 45.5        0.420368          0.413664       0.357217       0.493706              678.0                  1000       False            True
11                   11            0.107188                24               227                 49.0        0.410484          0.404891       0.349490       0.479948              678.0                  1000       False            True
10                   10            0.106569                26               227                 73.5        0.407970          0.403736       0.349526       0.471193              678.0                  1000        True            True
 8                    8           